# Bowling Score Detection from Video
## DSAI 352 — Computer Vision | Spring 2026

This notebook implements a full computer vision pipeline that:
1. Fine-tunes YOLOv8 on a custom bowling-pin dataset
2. Runs detection and tracking on a bowling video
3. Detects when each pin falls using movement analysis
4. Renders an annotated output video with timestamps and score

---

## Section 1 — Environment Setup

In [1]:
!pip install ultralytics roboflow supervision -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.2 MB/s eta 0:00:00


In [2]:
import cv2
import numpy as np
import os
import matplotlib.pyplot as plt
from collections import defaultdict
from ultralytics import YOLO
from google.colab import files
from google.colab.patches import cv2_imshow

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


---
## Section 2 — Video Ingestion and Preprocessing

In [3]:
uploaded = files.upload()
video_path_original = list(uploaded.keys())[0]
print(f"Video uploaded: {video_path_original}")

Saving 20260503_000448.mp4 to 20260503_000448.mp4
Video uploaded: 20260503_000448.mp4


In [4]:
cap = cv2.VideoCapture(video_path_original)
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
duration = total_frames / fps
cap.release()

print(f"FPS: {fps:.1f}")
print(f"Dimensions: {width} x {height}")
print(f"Total frames: {total_frames}")
print(f"Duration: {duration:.1f}s")

FPS: 59.9
Dimensions: 3840 x 2160
Total frames: 1772
Duration: 29.6s


In [5]:
output_small = "video_1080p.mp4"
cap = cv2.VideoCapture(video_path_original)
fps = cap.get(cv2.CAP_PROP_FPS)
new_width, new_height = 1920, 1080

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_small, fourcc, fps, (new_width, new_height))

print("Resizing video, please wait...")
while True:
    ret, frame = cap.read()
    if not ret:
        break
    out.write(cv2.resize(frame, (new_width, new_height)))

cap.release()
out.release()

video_path = output_small
print(f"Done. video_path set to: {video_path}")

Resizing video, please wait...
Done. video_path set to: video_1080p.mp4


---
## Section 3 — Dataset Preparation

We extract one frame per second from the video to create a labeled dataset for training.  
Frames are then uploaded to **Roboflow** for bounding-box annotation and exported in YOLOv8 format.

In [6]:
os.makedirs("frames", exist_ok=True)
cap = cv2.VideoCapture(video_path)
fps_val = cap.get(cv2.CAP_PROP_FPS)
extract_every_n = int(fps_val)

frame_count = 0
saved_count = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    if frame_count % extract_every_n == 0:
        cv2.imwrite(f"frames/frame_{saved_count:04d}.jpg", frame)
        saved_count += 1
    frame_count += 1
cap.release()
print(f"Frames extracted: {saved_count}")

Frames extracted: 31


In [7]:
import shutil
shutil.make_archive("frames", "zip", "frames")
files.download("frames.zip")
print("frames.zip downloaded. Upload to Roboflow for annotation.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

frames.zip downloaded. Upload to Roboflow for annotation.


In [8]:
from roboflow import Roboflow

from google.colab import userdata
rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
project = rf.workspace("tokas-workspace-33oqd").project("bowling-pins-6pci7")
version = project.version(1)
dataset = version.download("yolov8")
print("Dataset downloaded.")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to bowling-pins-1 in yolov8:: 100%|██████████| 67/67 [00:00<00:00, 5121.06it/s]

Dataset downloaded.


---
## Section 4 — Model Training (Fine-tuning YOLOv8)

We fine-tune a pretrained **YOLOv8s** model on our custom bowling-pin dataset.  
The pretrained weights (trained on COCO/ImageNet) provide a strong visual backbone;  
fine-tuning adapts the detection head to recognize our specific pin classes: `blue`, `red`, `yellow`.

In [9]:

model = YOLO("yolov8s.pt")

results = model.train(
    data="bowling-pins-1/data.yaml",
    epochs=50,
    imgsz=640,
    batch=8,
    name="bowling_pins_model",
    patience=10
)

print("Training complete.")

Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=bowling-pins-1/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=bowling_pins_model, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patien

---
## Section 5 — Detection and Tracking

We run the fine-tuned model on the full video using **ByteTrack** for multi-object tracking.  
Each pin is assigned a stable track ID across frames, and its bounding box and position are recorded.

In [10]:

model_path = "/content/runs/detect/bowling_pins_model/weights/best.pt"
model = YOLO(model_path)


results = model.track(
    source=video_path,
    tracker="bytetrack.yaml",
    conf=0.5,
    iou=0.5,
    persist=True,
    stream=True
)


tracking_data = []
fps_val = cv2.VideoCapture(video_path).get(cv2.CAP_PROP_FPS)

for frame_idx, result in enumerate(results):
    frame_info = {"frame_idx": frame_idx, "pins": []}
    if result.boxes is not None and result.boxes.id is not None:
        boxes  = result.boxes.xyxy.cpu().numpy()
        ids    = result.boxes.id.cpu().numpy()
        classes = result.boxes.cls.cpu().numpy()
        confs  = result.boxes.conf.cpu().numpy()
        for box, tid, cls, conf in zip(boxes, ids, classes, confs):
            x1, y1, x2, y2 = box
            w, h = x2 - x1, y2 - y1
            frame_info["pins"].append({
                "track_id": int(tid),
                "class": model.names[int(cls)],
                "conf": float(conf),
                "box": [float(x1), float(y1), float(x2), float(y2)],
                "aspect_ratio": h / w if w > 0 else 1.0
            })
    tracking_data.append(frame_info)

print(f"Tracking complete. Total frames processed: {len(tracking_data)}")

requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 695ms
Prepared 1 package in 289ms
Installed 1 package in 5ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 1.6s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


video 1/1 (frame 1/1772) /content/video_1080p.mp4: 384x640 2 blues, 4 reds, 2 yellows, 402.6ms
video 1/1 (frame 2/1772) /content/video_1080p.mp4: 384x640 2 blues, 4 reds, 2 yellows, 373.5ms
video 1/1 (frame 3/1772) /content/video_1080p.mp4: 384x640 2 blues, 4 reds, 2 yellows, 362.4ms
video 1/1 (frame 4/1772) /content/video_1080p.mp4: 384x640 2 blues, 4 reds, 2 yellows, 390.2ms
video 1/1 (frame 5/1772) /content/video_1080p.mp4: 384x640 2 blues, 5 reds, 2 yellows, 370.8ms
video 1/1 (frame 6/1772) /content/video_1080p.mp4: 384x640 2 blues, 4 reds, 2 yellows, 378.0ms
video 1/1 (frame 7/1772) /content/video_1080p.mp4: 384x640 2 blues, 4 r

In [11]:

class_ids = defaultdict(list)

for frame_info in tracking_data:
    for pin in frame_info["pins"]:
        tid, cls = pin["track_id"], pin["class"]
        if tid not in [x[1] for x in class_ids[cls]]:
            class_ids[cls].append((frame_info["frame_idx"], tid))

valid_ids = {}
for cls, id_list in class_ids.items():
    for _, tid in sorted(id_list)[:2]:
        valid_ids[tid] = cls

print("Valid pin IDs:")
for tid, cls in sorted(valid_ids.items()):
    print(f"  ID {tid} -> {cls}")

Valid pin IDs:
  ID 1 -> yellow
  ID 2 -> red
  ID 3 -> yellow
  ID 4 -> blue
  ID 5 -> blue
  ID 6 -> red


---
## Section 6 — Fall Detection

**Approach:** Movement-based fall detection.

Since the camera angle is top-down, aspect ratio alone cannot reliably distinguish standing vs. fallen pins (both appear roughly circular from above). Instead, we track the **center-point displacement** of each pin across frames:

- A **spike** in displacement (pin center moves suddenly) indicates the pin is being knocked over.
- If movement **stabilizes** after the spike (pin has come to rest), the fall is confirmed.

**Parameters:**
- `MOVEMENT_THRESHOLD = 10` px — minimum spike to consider
- `STABLE_FRAMES = 10` — frames checked after spike
- `STABLE_THRESHOLD = 8` px — maximum average movement to confirm stability

In [12]:

pin_centers = {}

for frame_info in tracking_data:
    for pin in frame_info["pins"]:
        tid = pin["track_id"]
        if tid not in valid_ids:
            continue
        x1, y1, x2, y2 = pin["box"]
        cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
        if tid not in pin_centers:
            pin_centers[tid] = []
        pin_centers[tid].append((frame_info["frame_idx"], cx, cy))

In [13]:

MOVEMENT_THRESHOLD = 10
STABLE_FRAMES      = 10
STABLE_THRESHOLD   = 8

pin_fall_status3 = {}
pin_fall_frame3  = {}

for tid in valid_ids:
    if tid not in pin_centers or len(pin_centers[tid]) < 2:
        continue
    frames_list, cxs, cys = zip(*pin_centers[tid])
    displacements = [0]
    for j in range(1, len(cxs)):
        dx, dy = cxs[j] - cxs[j-1], cys[j] - cys[j-1]
        displacements.append((dx**2 + dy**2) ** 0.5)

    fell = False
    for j in range(1, len(displacements) - STABLE_FRAMES):
        if displacements[j] > MOVEMENT_THRESHOLD:
            after_spike = displacements[j+1 : j+1+STABLE_FRAMES]
            if sum(after_spike) / len(after_spike) < STABLE_THRESHOLD:
                pin_fall_status3[tid] = "fallen"
                pin_fall_frame3[tid]  = frames_list[j]
                fell = True
                break
    if not fell:
        pin_fall_status3[tid] = "standing"


print(f"{'Pin ID':<10} {'Class':<10} {'Status':<10} {'Fell at (sec)'}")
print("-" * 45)
for tid in sorted(valid_ids):
    status = pin_fall_status3.get(tid, "standing")
    cls    = valid_ids[tid]
    if status == "fallen":
        print(f"{tid:<10} {cls:<10} {status:<10} {pin_fall_frame3[tid]/fps_val:.2f}s")
    else:
        print(f"{tid:<10} {cls:<10} {status:<10} -")

fallen_count = sum(1 for s in pin_fall_status3.values() if s == "fallen")
print(f"\nTotal pins fallen: {fallen_count}/6")

Pin ID     Class      Status     Fell at (sec)
---------------------------------------------
1          yellow     fallen     24.27s
2          red        standing   -
3          yellow     fallen     3.14s
4          blue       fallen     27.47s
5          blue       fallen     14.76s
6          red        fallen     1.87s

Total pins fallen: 5/6


---
## Section 7 — Annotated Video Rendering

We re-render the original video with the following overlays:
- **Bounding box** on each detected pin (color-coded by class)
- **Red X marker** on each pin from the moment it falls
- **Timestamp label** showing when the fall occurred
- **HUD** (top-left) showing current time and pins-down count
- **GAME OVER** summary in the final 3 seconds

In [14]:
output_video_path = "bowling_final.mp4"

cap = cv2.VideoCapture(video_path)
fps_val     = cap.get(cv2.CAP_PROP_FPS)
width       = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height      = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video_path, fourcc, fps_val, (width, height))

class_colors = {
    "blue":   (255, 100,   0),
    "red":    (  0,   0, 255),
    "yellow": (  0, 220, 255)
}

frame_idx = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break

    current_time = frame_idx / fps_val
    frame_info   = tracking_data[frame_idx] if frame_idx < len(tracking_data) else {"pins": []}

    total_fallen_now = sum(
        1 for tid, fall_frame in pin_fall_frame3.items()
        if frame_idx >= fall_frame
    )

    for pin in frame_info["pins"]:
        tid = pin["track_id"]
        if tid not in valid_ids:
            continue
        cls   = valid_ids[tid]
        color = class_colors.get(cls, (255, 255, 255))
        x1, y1, x2, y2 = [int(v) for v in pin["box"]]
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

        is_fallen = (
            tid in pin_fall_status3 and
            pin_fall_status3[tid] == "fallen" and
            frame_idx >= pin_fall_frame3[tid]
        )

        if is_fallen:
            fall_sec = pin_fall_frame3[tid] / fps_val
            cv2.line(frame, (x1, y1), (x2, y2), (0, 0, 255), 3)
            cv2.line(frame, (x1, y2), (x2, y1), (0, 0, 255), 3)
            cv2.putText(frame, f"PIN {tid} fell @ {fall_sec:.1f}s",
                        (x1, max(y1-15, 20)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,255), 2)
        else:
            cv2.putText(frame, f"PIN {tid} ({cls})",
                        (x1, max(y1-15, 20)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)


    cv2.putText(frame, f"Time: {current_time:.1f}s", (20, 45),
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255,255,255), 2)
    cv2.putText(frame, f"Pins Down: {total_fallen_now}/6", (20, 90),
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255,255,255), 2)


    if frame_idx >= total_frames - int(fps_val * 3):
        overlay = frame.copy()
        cv2.rectangle(overlay, (width//2-320, height//2-120),
                      (width//2+320, height//2+120), (0,0,0), -1)
        cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)
        cv2.putText(frame, "GAME OVER",
                    (width//2-160, height//2-50), cv2.FONT_HERSHEY_SIMPLEX, 1.8, (0,255,255), 3)
        cv2.putText(frame, f"Pins Down: {total_fallen_now}/6",
                    (width//2-180, height//2+20), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255,255,255), 2)
        cv2.putText(frame, f"Duration: {current_time:.1f}s",
                    (width//2-150, height//2+70), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255,255,255), 2)

    out.write(frame)
    frame_idx += 1

cap.release()
out.release()
print(f"Annotated video saved to: {output_video_path}")

Annotated video saved to: bowling_final.mp4


In [15]:
files.download("bowling_final.mp4")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>